# Testing MultiCoCo Attribute Forwarding Fix

This notebook tests the `__getattr__` fix for the MultiCoCo class to ensure proper attribute forwarding to the underlying model. This addresses the critical flaw where LatentWrapper couldn't access InternVL model attributes like `extract_feature`, `dtype`, `conv_template`, etc.

## Problem Summary

The original issue was:
1. **MultiCoCoRunner** wraps a `MultiCoCo` instance with `LatentWrapper`
2. **LatentWrapper** expects to access attributes like `extract_feature()`, `dtype`, `conv_template` on the base model
3. **MultiCoCo** is just a wrapper - these attributes exist on the underlying InternVL model (`self.model`)
4. **Without forwarding**, this would cause `AttributeError` and break multimodal functionality

## Solution

Added `__getattr__` method to MultiCoCo that forwards unknown attributes to the underlying model.

## Section 1: Mocking Model, Tokenizer, and ImageProcessor Classes

First, we'll create mock classes that implement only the minimal required attributes and methods for MultiCoCo testing, avoiding the need to download actual models from HuggingFace.

In [ ]:
import sys
import os
import torch
import torch.nn as nn
from typing import Any, Dict, List, Optional
from unittest.mock import Mock, MagicMock

# Add the multicoco package to the path
sys.path.append('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco')

print("🛠️ Creating mock classes for testing...")

class MockConfig:
    """Mock config that mimics InternVL config structure"""
    def __init__(self):
        self.vocab_size = 32000
        self.downsample_ratio = 0.5
        self.attn_implementation = 'sdpa'
        
        # Vision config for multimodal dimensions
        self.vision_config = MockVisionConfig()

class MockVisionConfig:
    """Mock vision config"""
    def __init__(self):
        self.hidden_size = 1024
        self.image_size = 448

class MockEmbedding(nn.Module):
    """Mock embedding layer"""
    def __init__(self):
        super().__init__()
        self.num_embeddings = 32000
        self.embedding_dim = 2048
        self.weight = torch.randn(32000, 2048, dtype=torch.bfloat16)
        self.padding_idx = None
        self.max_norm = None
        self.norm_type = 2.0
        self.scale_grad_by_freq = False
        self.sparse = False

class MockLanguageModel(nn.Module):
    """Mock language model that mimics InternVL's language_model"""
    def __init__(self):
        super().__init__()
        self.config = MockConfig()
        self._embedding = MockEmbedding()
    
    def get_input_embeddings(self):
        return self._embedding
    
    def resize_token_embeddings(self, new_num_tokens):
        print(f"Mock: Resizing embeddings to {new_num_tokens}")
        # Just update the mock
        self._embedding.num_embeddings = new_num_tokens
        self._embedding.weight = torch.randn(new_num_tokens, 2048, dtype=torch.bfloat16)

class MockModel(nn.Module):
    """
    Mock model that simulates InternVL with the key attributes LatentWrapper needs
    """
    def __init__(self):
        super().__init__()
        self.config = MockConfig()
        self.language_model = MockLanguageModel()
        self.dtype = torch.bfloat16
        self.img_context_token_id = None
        self.num_image_token = 256
        
        # Mock conversation template 
        self.conv_template = MockConvTemplate()
        
        # Initialize weights to have proper parameter access
        self.dummy_param = nn.Parameter(torch.randn(1, dtype=torch.bfloat16))
    
    def extract_feature(self, pixel_values):
        """Mock extract_feature method that LatentWrapper calls"""
        print(f"Mock: extract_feature called with shape {pixel_values.shape}")
        # Return mock vision embeddings
        batch_size = pixel_values.shape[0]
        return torch.randn(batch_size, 256, 1024, dtype=self.dtype)
    
    def get_input_embeddings(self):
        return self.language_model.get_input_embeddings()
    
    def forward(self, **kwargs):
        # Mock forward pass
        input_ids = kwargs.get('input_ids')
        if input_ids is not None:
            batch_size, seq_len = input_ids.shape
            vocab_size = 32000
            logits = torch.randn(batch_size, seq_len, vocab_size, dtype=self.dtype)
            return MockModelOutput(logits=logits)
        return MockModelOutput()
    
    def generate(self, **kwargs):
        # Mock generation
        input_ids = kwargs.get('input_ids')
        max_new_tokens = kwargs.get('max_new_tokens', 50)
        if input_ids is not None:
            batch_size, seq_len = input_ids.shape
            new_tokens = torch.randint(0, 32000, (batch_size, max_new_tokens))
            return torch.cat([input_ids, new_tokens], dim=1)
        return torch.randint(0, 32000, (1, max_new_tokens))

class MockConvTemplate:
    """Mock conversation template"""
    def get_prompt(self, question):
        return f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

class MockModelOutput:
    """Mock model output"""
    def __init__(self, logits=None):
        self.logits = logits

class MockTokenizer:
    """Mock tokenizer with essential methods"""
    def __init__(self):
        self.vocab_size = 32000
        self.pad_token = '<pad>'
        self.eos_token = '</s>'
        self.pad_token_id = 0
        self.eos_token_id = 1
        self.unk_token_id = 2
        
        # Mock vocab
        self._vocab = {
            '<pad>': 0,
            '</s>': 1,
            '<unk>': 2,
            '<IMG_CONTEXT>': 3,
            '<|start_latent|>': 4,
            '<|end_latent|>': 5,
            '<|latent|>': 6
        }
        
        # Track added tokens
        self._added_tokens = []
    
    def __len__(self):
        return self.vocab_size + len(self._added_tokens)
    
    def get_vocab(self):
        return self._vocab
    
    def convert_tokens_to_ids(self, token):
        return self._vocab.get(token, self.unk_token_id)
    
    def add_special_tokens(self, special_tokens_dict):
        """Mock adding special tokens"""
        additional_tokens = special_tokens_dict.get('additional_special_tokens', [])
        print(f"Mock: Adding {len(additional_tokens)} special tokens: {additional_tokens}")
        self._added_tokens.extend(additional_tokens)
        
        # Add to vocab with new IDs
        for i, token in enumerate(additional_tokens):
            if token not in self._vocab:
                self._vocab[token] = self.vocab_size + len(self._added_tokens) + i
    
    def encode(self, text, add_special_tokens=True, return_tensors=None):
        # Simple mock encoding
        tokens = [1, 2, 3, 4, 5]  # Mock token IDs
        if return_tensors == "pt":
            return torch.tensor([tokens])
        return tokens
    
    def decode(self, token_ids, skip_special_tokens=True):
        # Simple mock decoding
        return "Mock decoded text"

class MockImageProcessor:
    """Mock image processor"""
    def __init__(self):
        self.do_resize = True
        self.size = {"height": 448, "width": 448}

print("✅ Mock classes created successfully!")

## Section 2: Monkeypatching MultiCoCo Initialization to Use Mocks

Now we'll override MultiCoCo's `_initialize_components` method to return our mock objects instead of trying to load from HuggingFace.

In [ ]:
import sys
import os
import unittest.mock

# Add the multicoco directory to the path
sys.path.append('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco')

# Patch all HuggingFace components before importing
with unittest.mock.patch('transformers.AutoModel.from_pretrained') as mock_model, \
     unittest.mock.patch('transformers.AutoTokenizer.from_pretrained') as mock_tokenizer, \
     unittest.mock.patch('transformers.AutoImageProcessor.from_pretrained') as mock_image_processor:
    
    # Configure the mocks to return our mock objects
    mock_model.return_value = MockInternVLModel()
    mock_tokenizer.return_value = MockTokenizer()
    mock_image_processor.return_value = MockImageProcessor()
    
    # Now import MultiCoCo (this will use our mocked components)
    from multicoco.model import MultiCoCo
    
    print("Successfully imported MultiCoCo with mocked HuggingFace components!")
    print(f"Mock model class: {type(mock_model.return_value)}")
    print(f"Mock tokenizer class: {type(mock_tokenizer.return_value)}")
    print(f"Mock image processor class: {type(mock_image_processor.return_value)}")

## Section 3: Creating MultiCoCo Instance with Mock Model

Now we'll create a MultiCoCo instance using our monkeypatched initialization. This should work without any HuggingFace downloads and demonstrate that our recursion fix works.

In [ ]:
from multicoco.constants import COCONUT_SPECIAL_TOKENS

print("🚀 Creating MultiCoCo instance with mocked components...")

try:
    # Create MultiCoCo with mock components
    # This should NOT cause infinite recursion now
    multicoco_model = MultiCoCo(
        model_id="mock-model",  # This will be ignored due to our monkeypatch
        special_tokens=list(COCONUT_SPECIAL_TOKENS),
        torch_dtype="bfloat16"
    )
    
    print("✅ SUCCESS: MultiCoCo instance created without recursion errors!")
    print(f"   Model type: {type(multicoco_model)}")
    print(f"   Underlying model type: {type(multicoco_model.model)}")
    print(f"   Tokenizer type: {type(multicoco_model.tokenizer)}")
    print(f"   Image processor type: {type(multicoco_model.image_processor)}")
    
    # Verify it has the model attribute (no recursion on this access)
    assert hasattr(multicoco_model, 'model'), "MultiCoCo should have 'model' attribute"
    print("✅ Model attribute exists and is accessible")
    
except Exception as e:
    print(f"❌ FAILED: {e}")
    import traceback
    traceback.print_exc()

## Section 4: Testing Attribute Forwarding (__getattr__)

Now for the critical test: verifying that the `__getattr__` method correctly forwards attributes from MultiCoCo to the underlying mock model. This simulates what LatentWrapper needs to do.